## Amazon AgentCore Bedrock Code Interpreter를 사용한 고급 데이터 분석 - 튜토리얼(Strands)
이 튜토리얼에서는 Python 코드 실행을 통해 고급 데이터 분석을 수행하는 AI 에이전트를 만드는 방법을 알아봅니다. Amazon Bedrock AgentCore Code Interpreter를 사용하여 LLM이 생성한 코드를 실행합니다.

AgentCore Bedrock Code Interpreter를 사용하여 다음 작업을 수행합니다.
1. 샌드박스 환경 설정
2. 사용자 질의를 바탕으로 코드를 생성하여 고급 데이터 분석을 수행하는 Strands 기반 에이전트 구성
3. Code Interpreter를 사용하여 샌드박스 환경에서 코드 실행
4. 사용자에게 결과 표시

## 사전 요구 사항
- Bedrock AgentCore Code Interpreter에 액세스할 수 있는 AWS 계정
- Code Interpreter 리소스를 생성하고 관리하는 데 필요한 IAM 권한
- 필수 Python 패키지 설치(boto3, bedrock-agentcore 및 strands 포함)
- Amazon Bedrock의 모델을 호출할 수 있는 권한이 있는 IAM 역할
 - 미국 오리건(us-west-2) 리전의 Claude Haiku 4.5 및 Claude Sonnet 3.5 모델 액세스 권한

## IAM 실행 역할에 다음 IAM 정책을 연결해야 합니다

~~~ {
"Version": "2012-10-17",
"Statement": [
    {
        "Effect": "Allow",
        "Action": [
            "bedrock-agentcore:CreateCodeInterpreter",
            "bedrock-agentcore:StartCodeInterpreterSession",
            "bedrock-agentcore:InvokeCodeInterpreter",
            "bedrock-agentcore:StopCodeInterpreterSession",
            "bedrock-agentcore:DeleteCodeInterpreter",
            "bedrock-agentcore:ListCodeInterpreters",
            "bedrock-agentcore:GetCodeInterpreter"
        ],
        "Resource": "*"
    },
    {
        "Effect": "Allow",
        "Action": [
            "logs:CreateLogGroup",
            "logs:CreateLogStream",
            "logs:PutLogEvents"
        ],
        "Resource": "arn:aws:logs:*:*:log-group:/aws/bedrock-agentcore/code-interpreter*"
    }
]
}

## 작동 방식

코드 실행 샌드박스는 Code Interpreter, 셸, 파일 시스템을 갖춘 격리 환경을 생성하여 에이전트가 사용자 질의를 안전하게 처리할 수 있도록 합니다. 대규모 언어 모델(LLM)이 도구 선택을 지원한 후 이 세션 내에서 코드가 실행되며, 결과는 종합을 위해 사용자 또는 에이전트에게 반환됩니다.

![로컬 아키텍처](code-interpreter.png)

## 1. 환경 설정

먼저 필요한 라이브러리를 가져오고 Code Interpreter 클라이언트를 초기화합니다.

기본 세션 제한 시간은 900초(15분)입니다. 데이터를 상세히 분석할 예정이므로 세션 제한 시간을 1200초(20분)로 늘려 시작합니다.

In [ ]:
!pip install --upgrade -r requirements.txt

In [ ]:
from bedrock_agentcore.tools.code_interpreter_client import CodeInterpreter
from strands import Agent, tool
from strands.models import BedrockModel
import json
import pandas as pd
from typing import Dict, Any

# 지원되는 AWS 리전에서 Code Interpreter 초기화
code_client = CodeInterpreter("us-west-2")
code_client.start(session_timeout_seconds=1200)

## 2. 로컬 데이터 파일 읽기

이제 샘플 데이터 파일의 내용을 읽습니다. 이 파일은 Name, Preferred_City, Preferred_Animal, Preferred_Thing의 4개 열과 약 300,000개의 무작위 데이터 레코드로 구성되어 있습니다.

잠시 후 에이전트로 이 파일을 분석하여 분포와 이상치를 파악합니다.

In [ ]:
df_data = pd.read_csv("samples/data.csv")
df_data.head()

In [ ]:
def read_file(file_path: str) -> str:
    """오류 처리를 포함해 파일 내용을 읽는 헬퍼 함수"""
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            return file.read()
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        return ""
    except Exception as e:
        print(f"An error occurred: {e}")
        return ""


data_file_content = read_file("samples/data.csv")

## 3. 샌드박스 환경용 파일 준비

샌드박스 환경에서 생성할 파일을 정의하는 구조를 만듭니다.

In [ ]:
files_to_create = [{"path": "data.csv", "text": data_file_content}]

## 4. 도구 호출용 헬퍼 함수 생성

이 헬퍼 함수를 사용하면 샌드박스 도구를 더 쉽게 호출하고 응답을 처리할 수 있습니다. 활성 세션에서는 지원되는 언어(Python, JavaScript)로 코드를 실행하고, 종속성 구성에 따른 라이브러리에 액세스하고, 시각화를 생성하고, 실행 간 상태를 유지할 수 있습니다.

In [ ]:
def call_tool(tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    """샌드박스 도구를 호출하는 헬퍼 함수

    매개변수:
        tool_name (str): 호출할 도구 이름
        arguments (Dict[str, Any]): 도구에 전달할 인수

    반환값:
        Dict[str, Any]: JSON 형식의 결과
    """
    response = code_client.invoke(tool_name, arguments)
    for event in response["stream"]:
        return json.dumps(event["result"])

## 5. 코드 샌드박스에 데이터 파일 쓰기

이제 데이터 파일을 샌드박스 환경에 쓰고 정상적으로 생성되었는지 확인합니다.

In [ ]:
# 샌드박스에 파일 쓰기
writing_files = call_tool("writeFiles", {"content": files_to_create})
print("Writing files result:")
print(writing_files)

# 파일이 생성되었는지 확인
listing_files = call_tool("listFiles", {"path": ""})
print("\nFiles in sandbox:")
print(listing_files)

## 6. Strands 기반 에이전트를 사용한 고급 분석 수행

이제 위에서 샌드박스에 업로드한 데이터 파일을 분석하도록 에이전트를 구성합니다.

### 6.1 시스템 프롬프트 정의
AI 어시스턴트의 동작과 기능을 정의합니다. 항상 코드 실행과 데이터 기반 추론을 통해 답변을 검증하도록 지시합니다.

In [ ]:
SYSTEM_PROMPT = """You are a helpful AI assistant that validates all answers through code execution using the tools provided. DO NOT Answer questions without using the tools

VALIDATION PRINCIPLES:
1. When making claims about code, algorithms, or calculations - write code to verify them
2. Use execute_python to test mathematical calculations, algorithms, and logic
3. Create test scripts to validate your understanding before giving answers
4. Always show your work with actual code execution
5. If uncertain, explicitly state limitations and validate what you can

APPROACH:
- If asked about a programming concept, implement it in code to demonstrate
- If asked for calculations, compute them programmatically AND show the code
- If implementing algorithms, include test cases to prove correctness
- Document your validation process for transparency
- The sandbox maintains state between executions, so you can refer to previous results

TOOL AVAILABLE:
- execute_python: Run Python code and see output

RESPONSE FORMAT: The execute_python tool returns a JSON response with:
- sessionId: The sandbox session ID
- id: Request ID
- isError: Boolean indicating if there was an error
- content: Array of content objects with type and text/data
- structuredContent: For code execution, includes stdout, stderr, exitCode, executionTime

For successful code execution, the output will be in content[0].text and also in structuredContent.stdout.
Check isError field to see if there was an error.

Be thorough, accurate, and always validate your answers when possible."""

### 6.2 코드 실행 도구 정의
다음으로 코드 샌드박스에서 코드를 실행할 때 에이전트가 사용할 함수를 도구로 정의합니다. @tool 데코레이터를 사용하여 이 함수를 에이전트의 사용자 정의 도구로 지정합니다.

활성 Code Interpreter 세션에서는 지원되는 언어(Python, JavaScript)로 코드를 실행하고, 종속성 구성에 따른 라이브러리에 액세스하고, 시각화를 생성하고, 실행 간 상태를 유지할 수 있습니다.

In [ ]:
# Code Interpreter 도구 정의 및 구성
@tool
def execute_python(code: str, description: str = "") -> str:
    """Execute Python code in the sandbox."""

    if description:
        code = f"# {description}\n{code}"

    # 실행할 생성 코드 출력
    print(f"\n Generated Code: {code}")

    # 초기화된 Code Interpreter 세션에서 invoke 메서드를 호출하여 생성된 코드 실행
    response = code_client.invoke("executeCode", {"code": code, "language": "python", "clearContext": False})
    for event in response["stream"]:
        return json.dumps(event["result"])

### 6.3 에이전트 구성
Strands SDK를 사용하여 에이전트를 생성하고 구성합니다. 생성된 코드를 실행할 수 있도록 위에서 정의한 시스템 프롬프트와 도구를 에이전트에 제공합니다.

Claude Haiku 4.5 모델을 사용하고 [Cross-region Inference (CRIS)](https://docs.aws.amazon.com/bedrock/latest/userguide/cross-region-inference.html) 프로필 ID를 지정합니다.

In [ ]:
model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(model_id=model_id)

# 모델과 도구를 포함하여 Strands 에이전트 구성
agent = Agent(
    model=model,
    tools=[execute_python],
    system_prompt=SYSTEM_PROMPT,
    callback_handler=None,
)

## 7. 에이전트 호출 및 응답 처리
질의로 에이전트를 호출하고 에이전트의 응답을 처리합니다.


참고: 비동기 실행은 비동기 환경에서 수행해야 합니다.

## 7.1 탐색적 데이터 분석(EDA)을 수행하는 질의

먼저 에이전트에게 코드 샌드박스 환경의 데이터 파일을 탐색적으로 분석하도록 지시하는 질의를 사용합니다.

In [ ]:
query = "Load the file 'data.csv' and perform exploratory data analysis(EDA) on it. Tell me about distributions and outlier values."

# 에이전트를 비동기식으로 호출하고 응답 스트리밍
response_text = ""
async for event in agent.stream_async(query):
    if "data" in event:
        # 텍스트 응답 스트리밍
        chunk = event["data"]
        response_text += chunk
        print(chunk, end="")

## 7.2 정보를 추출하는 질의

이제 에이전트에게 코드 샌드박스 환경의 데이터 파일에서 특정 정보를 추출하도록 지시합니다.

In [ ]:
query = "Within the file 'data.csv', how many individuals with the first name 'Kimberly' have 'Crocodile' as their favourite animal?"

# 에이전트를 비동기식으로 호출하고 응답 스트리밍
response_text = ""
async for event in agent.stream_async(query):
    if "data" in event:
        # 텍스트 응답 스트리밍
        chunk = event["data"]
        response_text += chunk
        print(chunk, end="")

## 8. 정리

마지막으로 Code Interpreter 세션을 중지하여 정리합니다. 세션 사용이 끝나면 리소스를 해제하고 불필요한 비용이 발생하지 않도록 세션을 중지해야 합니다.

In [ ]:
# Code Interpreter 세션 중지
code_client.stop()
print("Code Interpreter session stopped successfully!")